In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/081d1e25-3f06-46a1-95ef-d10a62faa80f___GHLB2 Leaf 8600_final_masked.jpg
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/e6703c22-86f7-4bdf-9733-072180761976___RS_Late.B 5522_final_masked.jpg
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/9f946f43-424a-41be-9f91-b14fa3f2b37a___GHLB2 Leaf 8892_final_masked.jpg
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/dfc45d56-7ecb-46a5-b9da-9297df393f7b___RS_Late.B 6928_final_masked.jpg
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/84b7d3ab-56ae-4eef-a852-7b32f3ef0160___GHLB2 Leaf 150_final_masked.jpg
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented/Tomato___Late_blight/a5fe1111-38b0-4206-9ac7-3eec701beb79___GHLB2 Leaf 8868_final_masked.jpg
/kaggle/input/datasets/abdallah

In [2]:
!pip install timm

In [3]:
import torch
import torch.nn as nn
import timm
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
!kaggle datasets download -d vishnu23f3003285/comp-data

Dataset URL: https://www.kaggle.com/datasets/vishnu23f3003285/comp-data
License(s): MIT
100%|█████████████████████████████████████████| 836M/836M [00:04<00:00, 199MB/s]



In [6]:
!unzip comp-data.zip

Archive:  comp-data.zip
  inflating: data/cropped_testing/Potato_Early_blight/Pasted image (2).png  
  inflating: data/cropped_testing/Potato_Early_blight/Pasted image (3).png  
  inflating: data/cropped_testing/Potato_Early_blight/Pasted image.png  
  inflating: data/cropped_testing/Potato_Early_blight/images (100).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (11).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (12).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (13).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (14).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (15).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (16).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (17).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (18).jpg  
  inflating: data/cropped_testing/Potato_Early_blight/images (19).jpg  
  inflating: data/cropped_te

In [7]:
class_mapping = {
    "Potato_Early_blight": 0,
    "Potato_Lateblight": 1,
    "Potato_healthy": 2,
    "Tomato_Bacterial_spot": 3,
    "Tomato_Early_blight": 4,
    "Tomato_Late_blight": 5,
    "Tomato_Leaf_mold": 6,
    "Tomato_Septoria_leaf_spot": 7,
    "Tomato_Tomato_Yellow_Leaf_Curl_Virus": 8,
    "Tomato_healthy": 9,

    # PlantVillage
    "Potato___Early_blight": 0,
    "Potato___Late_blight": 1,
    "Potato___healthy": 2,
    "Tomato___Bacterial_spot": 3,
    "Tomato___Early_blight": 4,
    "Tomato___Late_blight": 5,
    "Tomato___Leaf_Mold": 6,
    "Tomato___Septoria_leaf_spot": 7,
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": 8,
    "Tomato___healthy": 9,
}

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [9]:
class PlantDocDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                for img in os.listdir(class_path):
                    self.samples.append(
                        (os.path.join(class_path, img),
                         class_mapping[class_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [53]:
plantdoc_root = "/kaggle/working/data/train"

full_pd_dataset = PlantDocDataset(
    plantdoc_root,
    transform=train_transform
)

indices = list(range(len(full_pd_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

pd_train_subset = Subset(full_pd_dataset, train_idx)

pd_val_dataset = PlantDocDataset(
    plantdoc_root,
    transform=val_transform
)

pd_val_subset = Subset(pd_val_dataset, val_idx)

print("PlantDoc train size:", len(pd_train_subset))
print("PlantDoc val size:", len(pd_val_subset))

PlantDoc train size: 2662
PlantDoc val size: 666


In [54]:
class PlantVillageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                valid_extensions = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

                for img in os.listdir(class_path):
                    if img.endswith(valid_extensions):
                        self.samples.append(
                            (os.path.join(class_path, img),
                             class_mapping[class_name])
                        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        if self.transform:
            image = self.transform(image)

        return image, label

In [55]:
plantvillage_root = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented"

pv_dataset = PlantVillageDataset(
    plantvillage_root,
    transform=train_transform
)

print("PlantVillage size:", len(pv_dataset))

PlantVillage size: 16859


In [64]:
# joint_train_dataset = ConcatDataset([pv_dataset, pd_train_subset])
joint_train_dataset = pd_train_subset


train_loader = DataLoader(
    joint_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    pd_val_subset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

num_classes = 10

In [65]:
model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=True,
    img_size=224
)

for param in model.parameters():
    param.requires_grad = False

for param in model.blocks[-2:].parameters():
    param.requires_grad = True

in_features = model.num_features

model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

In [66]:
from collections import Counter

train_labels = [label for _, label in pd_train_subset]
class_counts = Counter(train_labels)

weights = [1.0 / class_counts[i] for i in range(num_classes)]
weights = torch.tensor(weights).float().to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [67]:
from torch.amp import GradScaler, autocast

def train_model(model, train_loader, val_loader, epochs=30, patience=10):

    scaler = GradScaler("cuda")
    best_acc = 0
    early_stop = 0

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                with autocast("cuda"):
                    outputs = model(images)

                _, preds = torch.max(outputs,1)

                total += labels.size(0)
                correct += (preds==labels).sum().item()

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_dino_finetuned.pth")
            early_stop = 0
        else:
            early_stop += 1

        if early_stop >= patience:
            print("Early stopping triggered.")
            break

        scheduler.step()

    print("Best Validation Accuracy:", best_acc)

In [68]:
train_model(model, train_loader, val_loader, epochs=30)

Epoch 1: Loss=1.4898 | Val Acc=0.6411
Epoch 2: Loss=0.9883 | Val Acc=0.6652
Epoch 3: Loss=0.8529 | Val Acc=0.6847
Epoch 4: Loss=0.7417 | Val Acc=0.6877
Epoch 5: Loss=0.6514 | Val Acc=0.6907
Epoch 6: Loss=0.5747 | Val Acc=0.6441
Epoch 7: Loss=0.5049 | Val Acc=0.6922
Epoch 8: Loss=0.4705 | Val Acc=0.6907
Epoch 9: Loss=0.4206 | Val Acc=0.6952
Epoch 10: Loss=0.3939 | Val Acc=0.6562
Epoch 11: Loss=0.3602 | Val Acc=0.6727
Epoch 12: Loss=0.3447 | Val Acc=0.6697
Epoch 13: Loss=0.3357 | Val Acc=0.6381
Epoch 14: Loss=0.3328 | Val Acc=0.6517
Epoch 15: Loss=0.3104 | Val Acc=0.6577
Epoch 16: Loss=0.2893 | Val Acc=0.6532
Epoch 17: Loss=0.2875 | Val Acc=0.6562
Epoch 18: Loss=0.2783 | Val Acc=0.6832
Epoch 19: Loss=0.2748 | Val Acc=0.6742
Early stopping triggered.
Best Validation Accuracy: 0.6951951951951952


In [69]:
test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

plantdoc_test_root = "/kaggle/working/data/test"

pd_test_dataset = PlantDocDataset(
    plantdoc_test_root,
    transform=test_transform
)

test_loader = DataLoader(
    pd_test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("PlantDoc Test size:", len(pd_test_dataset))

PlantDoc Test size: 842


In [70]:
import timm
import torch.nn as nn

num_classes = 10

model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    img_size=224
)

in_features = model.num_features
model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)
model.load_state_dict(
    torch.load("best_dino_finetuned.pth", map_location=device)
)

model = model.to(device)
model.eval()

print("Best DINO model loaded successfully.")

Best DINO model loaded successfully.


In [71]:
from sklearn.metrics import confusion_matrix, classification_report

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print("\n==============================")
print("DINOv2 Test Accuracy:", test_acc)
print("==============================")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

class_names = [
     "Potato_Early_blight",
    "Potato_Lateblight",
    "Potato_healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_healthy",
]

print("\nPer Class Accuracy:")

for i, class_name in enumerate(class_names):
    class_total = cm[i].sum()
    class_correct = cm[i][i]

    acc = class_correct / class_total if class_total > 0 else 0
    print(f"{class_name}: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



DINOv2 Test Accuracy: 0.6923990498812351

Confusion Matrix:
[[51 15  0  8  8  2  1  2  0  0]
 [27 64  2  0  3  7  0  0  1  0]
 [ 2  1 60  0  0  0  0  0  0  0]
 [ 1  1  3 74  2  1  1 16  0  1]
 [ 5  0  0  9 33  7  2  6  0  0]
 [ 0  5  0  5 11 61  3  1  1  3]
 [ 0  0  0  9  4  5 55  0  0  0]
 [ 3  0  0 31 15  3  1 53  2  0]
 [ 0  0  0  5  1  0  3  0 72  3]
 [ 0  0  2  3  0  1  0  0  5 60]]

Per Class Accuracy:
Potato_Early_blight: 0.5862
Potato_Lateblight: 0.6154
Potato_healthy: 0.9524
Tomato_Bacterial_spot: 0.7400
Tomato_Early_blight: 0.5323
Tomato_Late_blight: 0.6778
Tomato_Leaf_mold: 0.7534
Tomato_Septoria_leaf_spot: 0.4907
Tomato_Tomato_Yellow_Leaf_Curl_Virus: 0.8571
Tomato_healthy: 0.8451

Classification Report:
                                      precision    recall  f1-score   support

                 Potato_Early_blight       0.57      0.59      0.58        87
                   Potato_Lateblight       0.74      0.62      0.67       104
                      Potato_healthy   